## Vectorization

In [6]:
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.normalizers import Lowercase

sample1 = "We are learning AI"
sample2 = "AI is a CS topic"

hf_tokenizer = Tokenizer(WordLevel(unk_token="<unk>"))
hf_tokenizer.normalizer = Lowercase()
hf_tokenizer.pre_tokenizer = Whitespace()

def tokenize_only(text: str):
    text_norm = hf_tokenizer.normalizer.normalize_str(text)
    return [tok for tok, _ in hf_tokenizer.pre_tokenizer.pre_tokenize_str(text_norm)]  # split

print(tokenize_only(sample1))
print(tokenize_only(sample2))


['we', 'are', 'learning', 'ai']
['ai', 'is', 'a', 'cs', 'topic']


In [7]:
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.normalizers import Lowercase

sample1 = "We are learning AI"
sample2 = "AI is a CS topic"
data = [sample1, sample2]

hf_tokenizer = Tokenizer(WordLevel(unk_token="<unk>"))
hf_tokenizer.normalizer = Lowercase()
hf_tokenizer.pre_tokenizer = Whitespace()

vocab_size = 8
trainer = WordLevelTrainer(
    vocab_size=vocab_size,
    special_tokens=["<unk>", "<pad>"]
)

hf_tokenizer.train_from_iterator(data, trainer)

vocab = hf_tokenizer.get_vocab()
vocab


{'<pad>': 1,
 'a': 3,
 '<unk>': 0,
 'is': 6,
 'cs': 5,
 'are': 4,
 'learning': 7,
 'ai': 2}

In [11]:
print(sample1)

enc1 = hf_tokenizer.encode(sample1)
print(enc1.tokens)
print(enc1.ids)

We are learning AI
['<unk>', 'are', 'learning', 'ai']
[0, 4, 7, 2]


In [12]:
print(sample2)

enc2 = hf_tokenizer.encode(sample2)
print(enc2.tokens)
print(enc2.ids)

AI is a CS topic
['ai', 'is', 'a', 'cs', '<unk>']
[2, 6, 3, 5, 0]


## Embedding

In [ ]:
import torch
import torch.nn as nn

vocab_size = 8
embed_dim = 4
embedding = nn.Embedding(vocab_size, embed_dim)

custom_weights = torch.tensor( [[-0.1882,  0.5530,  1.6267,  0.7013],
                                [ 1.7840, -0.8278, -0.2701,  1.3586],
                                [ 1.0281, -1.9094,  0.3182,  0.4211],
                                [-1.3083, -0.0987,  0.7647, -0.3680],
                                [ 0.2293,  1.3255,  0.1318,  2.0501],
                                [ 0.4058, -0.6624, -0.8745,  0.7203],
                                [ 0.5582,  0.0786, -0.6817,  0.6902],
                                [ 0.4309, -1.3067, -0.8823,  1.5977]]).float()
embedding.weight = nn.Parameter(custom_weights)
print(embedding.weight)

Parameter containing:
tensor([[-0.1882,  0.5530,  1.6267,  0.7013],
        [ 1.7840, -0.8278, -0.2701,  1.3586],
        [ 1.0281, -1.9094,  0.3182,  0.4211],
        [-1.3083, -0.0987,  0.7647, -0.3680],
        [ 0.2293,  1.3255,  0.1318,  2.0501],
        [ 0.4058, -0.6624, -0.8745,  0.7203],
        [ 0.5582,  0.0786, -0.6817,  0.6902],
        [ 0.4309, -1.3067, -0.8823,  1.5977]], requires_grad=True)


## Update

In [ ]:
import torch
import torch.nn as nn

vocab_size = 8
embed_dim = 4
embedding = nn.Embedding(vocab_size,
                         embed_dim)

custom_weights = torch.tensor( [[-0.1882,  0.5530,  1.6267,  0.7013],
                                [ 1.7840, -0.8278, -0.2701,  1.3586],
                                [ 1.0281, -1.9094,  0.3182,  0.4211],
                                [-1.3083, -0.0987,  0.7647, -0.3680],
                                [ 0.2293,  1.3255,  0.1318,  2.0501],
                                [ 0.4058, -0.6624, -0.8745,  0.7203],
                                [ 0.5582,  0.0786, -0.6817,  0.6902],
                                [ 0.4309, -1.3067, -0.8823,  1.5977]]).float()
embedding.weight = nn.Parameter(custom_weights)
print(embedding.weight)

Parameter containing:
tensor([[-0.1882,  0.5530,  1.6267,  0.7013],
        [ 1.7840, -0.8278, -0.2701,  1.3586],
        [ 1.0281, -1.9094,  0.3182,  0.4211],
        [-1.3083, -0.0987,  0.7647, -0.3680],
        [ 0.2293,  1.3255,  0.1318,  2.0501],
        [ 0.4058, -0.6624, -0.8745,  0.7203],
        [ 0.5582,  0.0786, -0.6817,  0.6902],
        [ 0.4309, -1.3067, -0.8823,  1.5977]], requires_grad=True)


In [ ]:
data = torch.tensor([0, 4, 7, 2, 1], dtype=torch.long)
label = torch.tensor([1], dtype=torch.float)

x = embedding(data)
x = nn.Flatten(0)(x)
x = nn.Linear(20, 1)(x)
output = nn.Sigmoid()(x)

In [ ]:
loss = nn.BCELoss()(output, label)
optimizer = torch.optim.Adam(embedding.parameters())
loss.backward()
optimizer.step()
print(embedding.weight)

Parameter containing:
tensor([[-0.1872,  0.5540,  1.6277,  0.7023],
        [ 1.7850, -0.8288, -0.2691,  1.3576],
        [ 1.0271, -1.9084,  0.3192,  0.4201],
        [-1.3083, -0.0987,  0.7647, -0.3680],
        [ 0.2283,  1.3265,  0.1308,  2.0491],
        [ 0.4058, -0.6624, -0.8745,  0.7203],
        [ 0.5582,  0.0786, -0.6817,  0.6902],
        [ 0.4319, -1.3057, -0.8813,  1.5967]], requires_grad=True)
